# TabFM v1.0.0 and TabFM-Ensemble

For instructions on how to install, dependencies and use TabFM, consult https://github.com/google-research/tabfm

In [ ]:
from pathlib import Path
import gc
import pickle
import sys
import time
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import torch
import safetensors
import tabfm
from sklearn.model_selection import StratifiedKFold
from tabfm import TabFMClassifier
from tabfm import tabfm_v1_0_0_pytorch

SEED = 94
DEVICE = 'cuda'

## Import data

In [ ]:
project_path = Path('/home/boscoll/Projects/cdk_predict')
train_test_split = project_path / 'data/development_cohort/train_test_split/genomic_train_test_split_26Feb_2026'
clinical_path = project_path / 'data/development_cohort/clinical/extended_clinical_features_cdk_1L_26Feb.csv'


with open(train_test_split, 'rb') as f:
    X_train_genomic, X_holdout_locked, y_train_surv, y_holdout_locked = pickle.load(f)
y_train_surv = y_train_surv.loc[X_train_genomic.index, ['Time', 'Event']].copy()

clinical = pd.read_csv(clinical_path, index_col=0)

#from the clinical only dataframe, remove the 'met_sample' variable
X_train_clinical = clinical.loc[X_train_genomic.index].copy()
X_train_clinical.drop(columns='met_sample', inplace=True)

X_train = pd.concat([clinical, X_train_genomic], axis=1) 

feature_sets = {
    'clinical': X_train_clinical,
    'genomic': X_train_genomic,
    'clinicogenomic': X_train,
}

In [ ]:
def twelve_month_target(y):
    event_by_12 = y['Event'].astype(bool) & y['Time'].le(12)
    known_at_12 = event_by_12 | y['Time'].ge(12)
    return event_by_12.astype('int8'), known_at_12

In [ ]:
torch.cuda.set_device(0)

foundation_model = tabfm_v1_0_0_pytorch.load(
    model_type='classification',
    device=DEVICE,
)

model_configs = {
    'tabfm_v1_0_0': 'base',
    'tabfm_v1_0_0_ensemble': 'ensemble',
}
score_names = [
    f'{feature_block}__{model_name}'
    for feature_block in feature_sets
    for model_name in model_configs
]

Using: NVIDIA A100 80GB PCIe


Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 515.05it/s]


Loading weights from local directory


In [ ]:

oof_score = pd.DataFrame(np.nan, index=X_train.index, columns=score_names)
fold_rows = []

splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
y_12_all, _ = twelve_month_target(y_train_surv)

for fold, (train_position, validation_position) in enumerate(
    splitter.split(X_train, y_12_all), start=1
):
    train_index = X_train.index[train_position]
    validation_index = X_train.index[validation_position]

    y_12, known_at_12 = twelve_month_target(y_train_surv.loc[train_index])
    eligible_train_index = train_index[known_at_12.to_numpy()]

    for feature_block, X_model in feature_sets.items():
        X_fold_train = X_model.loc[eligible_train_index]
        y_fold_train = y_12.loc[eligible_train_index]
        X_fold_validation = X_model.loc[validation_index]

        for model_name, configuration in model_configs.items():
            score_name = f'{feature_block}__{model_name}'
            common_options = dict(
                n_estimators=32,
                max_num_features=500,
                max_num_rows=None,
                random_state=SEED + fold,
                batch_size=1,
                verbose=False,
            )
            if configuration == 'ensemble':
                clf = TabFMClassifier.ensemble(foundation_model, **common_options)
            else:
                clf = TabFMClassifier(model=foundation_model, **common_options)

            started = time.perf_counter()
            clf.fit(X_fold_train, y_fold_train)
            positive_class_position = int(np.flatnonzero(clf.classes_ == 1)[0])
            oof_score.loc[validation_index, score_name] = clf.predict_proba(
                X_fold_validation
            )[:, positive_class_position]
            fold_rows.append({
                'fold': fold,
                'model': score_name,
                'n_training_rows': len(X_fold_train),
                'seconds': time.perf_counter() - started,
            })
            del clf
            gc.collect()
            torch.cuda.empty_cache()

fold_audit = pd.DataFrame(fold_rows)

## Save 12-month risks

In [ ]:
oof_risk = oof_score.mul(100)
oof_risk.index.name = 'record_id'
# oof_risk.to_csv('/home/boscoll/Projects/cdk_predict/results/model_benchmark_aug2026/tabfm_risks.csv)

# Pooled-cohort, cross-fitted TabFM model-reliance analysis

In [ ]:
RELIANCE_MUTATIONS = [
    'BRCA1_tsg', 'BRCA2_tsg', 'PTEN_tsg','RB1_tsg', 'PALB2_tsg',
]

X_genomic_all = pd.concat(
    [X_train_genomic, X_holdout_locked], axis=0
).replace([np.inf, -np.inf], np.nan).astype('float32')
X_genomic_all = X_genomic_all.loc[
    :, ~X_genomic_all.columns.str.contains('fga', case=False)
]
y_reliance = pd.concat([
    y_train_surv[['Time', 'Event']],
    y_holdout_locked[['Time', 'Event']],
], axis=0).loc[X_genomic_all.index]

X_clinical_all = clin.drop(columns='met_sample').loc[X_genomic_all.index]
X_reliance = pd.concat(
    [X_clinical_all, X_genomic_all], axis=1
).astype('category')

missing_genes = sorted(set(RELIANCE_MUTATIONS) - set(X_reliance.columns))


In [ ]:
reliance_rows = []

for gene_nr, gene in enumerate(tqdm(RELIANCE_MUTATIONS, desc='Alterations')):
    carrier_status = X_reliance[gene].astype('float32').astype(int)

    for repeat in tqdm(
        range(30), desc=gene, leave=False
    ):
        outer_cv = StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=SEED + repeat,
        )

        for outer_fold, (train_pos, test_pos) in enumerate(
            outer_cv.split(X_reliance, carrier_status), start=1
        ):
            seed = SEED + gene_nr * 10_000 + repeat * 10 + outer_fold
            train_index = X_reliance.index[train_pos]
            test_index = X_reliance.index[test_pos]

            y_12, known_at_12 = twelve_month_target(
                y_reliance.loc[train_index]
            )
            eligible_train_index = train_index[known_at_12.to_numpy()]
            X_fold_train = X_reliance.loc[eligible_train_index]
            y_fold_train = y_12.loc[eligible_train_index]
            if y_fold_train.nunique() != 2:
                raise ValueError(
                    f'{gene}, repeat {repeat}, fold {outer_fold}: '
                    'the eligible training target does not contain both classes'
                )

            carrier_index = test_index[
                carrier_status.loc[test_index].to_numpy() == 1
            ]
            X_observed = X_reliance.loc[carrier_index].copy()
            X_ablated = X_observed.copy()
            X_ablated[gene] = 0

            clf = TabFMClassifier(
                model=foundation_model,
                n_estimators=32,
                max_num_features=500,
                max_num_rows=None,
                random_state=seed,
                batch_size=1,
                verbose=False,
            )
            clf.fit(X_fold_train, y_fold_train)
            positive_class_position = int(
                np.flatnonzero(clf.classes_ == 1)[0]
            )
            risk_observed = (
                clf.predict_proba(X_observed)[:, positive_class_position] * 100
            )
            risk_ablated = (
                clf.predict_proba(X_ablated)[:, positive_class_position] * 100
            )

            for record_id, observed, ablated in zip(
                carrier_index, risk_observed, risk_ablated
            ):
                delta = observed - ablated
                reliance_rows.append({
                    'record_id': record_id,
                    'gene': gene,
                    'model': 'TabFM',
                    'repeat': repeat,
                    'outer_fold': outer_fold,
                    'risk_observed': observed,
                    'risk_ablated': ablated,
                    'delta': delta,
                    'absolute_delta': abs(delta),
                })
            del clf
            gc.collect()
            torch.cuda.empty_cache()

reliance_raw = pd.DataFrame(reliance_rows)
# reliance_raw.to_csv('/home/boscoll/Projects/cdk_predict/results/model_benchmark_aug2026/tabfm_model_reliance/reliance_all_repeats.csv', index=False)